In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from scipy.stats import spearmanr
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

sys.path.append("..")
from modeling import COVARIATES, N_SPLITS, build_lgbm, preprocess_fold

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train_diversity"
OUT_DIR = TRAIN_DIR / "alpha_diversity_shap"
TARGETS = ["Richness", "Shannon_diversity"]

model_input = pd.read_csv(TRAIN_DIR / "model_input_diet_oral_diversity_baseline.csv", low_memory=False)
features = COVARIATES + [c for c in model_input.columns if c not in ["participant_id", *COVARIATES, *TARGETS]]
x = model_input[features]

In [ ]:
mean_abs_shap, spearman = {}, {}
for target in TARGETS:
    y = model_input[target].to_numpy(dtype=float)
    values = np.full(x.shape, np.nan)
    for train, test in KFold(n_splits=N_SPLITS, shuffle=False).split(x):
        x_train, x_test = preprocess_fold(x.iloc[train], x.iloc[test])
        y_scaler = StandardScaler().fit(y[train].reshape(-1, 1))
        model = build_lgbm().fit(x_train, y_scaler.transform(y[train].reshape(-1, 1)).ravel())
        values[test] = shap.TreeExplainer(model).shap_values(x_test)
    mean_abs_shap[target] = np.abs(values).mean(axis=0)
    spearman[target] = [spearmanr(x[f], y, nan_policy="omit").correlation if x[f].nunique() > 1 else np.nan for f in features]

mean_abs_shap = pd.DataFrame(mean_abs_shap, index=features)
spearman = pd.DataFrame(spearman, index=features)
OUT_DIR.mkdir(parents=True, exist_ok=True)
mean_abs_shap.to_csv(OUT_DIR / "alpha_diversity_mean_abs_shap_baseline.csv")
spearman.to_csv(OUT_DIR / "alpha_diversity_spearman_feature_target_correlation_baseline.csv")
(mean_abs_shap * np.sign(spearman).fillna(0)).to_csv(OUT_DIR / "alpha_diversity_directional_mean_abs_shap_baseline.csv")
print(mean_abs_shap.mean(axis=1).sort_values(ascending=False).head(20))